# Regression

This assignment is based on the chapter Single Layer Networks: Regression from [Bishop book](bishopbook.com).

---
In this exercise, you will implement a traditional Machine Learning Regression pipeline from scratch.

## Problem
We want to predict a function, given by a mixture of Gaussians. The data is sampled using a lognormal noise distribution given by

$$\frac{1}{x\sigma\sqrt{2\pi}} \exp\left( -\frac{(\ln x - \mu)^2}{2\sigma^2} \right)$$

The `generate_data` function takes an argument, `n_samples`, and returns those many datapoints. On a real dataset, you won't know the nature of the function, hence you are not allowed to use any of the following variables/functions in Parts 1 and 2 of your code -- `means, stds, weights, get_mixture_density`.

The problem statement is to describe the unknown function, using the simplest model, which can still capture the complexity of the function. For now, you can just eye-ball your solutions to know whether you have acheived satisfactory performance.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Parameters for 3 Gaussians
means = [25, 58, 75]
stds = [8, 12, 5]
weights = [0.25, 0.3, 0.2]

def generate_data(n_samples):
    x_samples = np.linspace(0, 100, n_samples)
    x_samples = (x_samples) / 100.0   # NORMALIZATION

    def get_mixture_density(x_vals):
        return 100 * sum(w * (1 / (s * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_vals - m) / s) ** 2) 
                   for m, s, w in zip(means, stds, weights))

    y_true = get_mixture_density(x_samples*100.0)

    # 3. Add lognormal noise
    noise = np.random.lognormal(mean=0, sigma=0.1, size=n_samples)

    y_samples = y_true + noise
    
    # Return as a 2D array (n_samples, 2)
    return np.column_stack((x_samples, y_samples))

# Example usage
data = generate_data(100)
plt.scatter(data[:, 0], data[:,1])


---
## Part 1 | Setup

### Q1
Using the fact that the noise follows a lognormal distribution, derive an expression for the MLE, and consequently, choose an appropriate loss function to minimize. 

### Q2
Now, choose an appropriate set of basis functions. This is called feature engineering (a concept that has lost favor with the advent of deep neural networks, which are considered **universal function approximators**). For most problems, the best choice is $1, x, x^2, ..., x^n$, where $n$ is a hyperparameter, that signifies model complexity. 

### Q3
Using decision theory, compute $\mathbb{E}[t|\boldsymbol{x}]$, the predicted y. To do so, use calculus of variations to compute $\frac{\delta \mathbb{E}[L]}{\delta f(\mathbf{x})}$ and equate it to 0. You can assume $L$ coressponds to L2 loss. 

---

# Part 1 — Theory

## Q1. MLE for lognormal noise



Assume the noise $\eta$ follows a lognormal distribution:
$$
p_\eta(z)
= \frac{1}{z\,\sigma\sqrt{2\pi}}
\exp\!\left(-\frac{(\ln z - \mu)^2}{2\sigma^2}\right),
\qquad z>0.
$$

The observation model is
$$
t = f(x) + \eta.
$$

For a dataset $\{(x_i,t_i)\}_{i=1}^N$, the log-likelihood is
$$
\ell(f)
= \sum_{i=1}^N \log p_\eta\!\left(t_i - f(x_i)\right).
$$

Substituting the lognormal density,
$$
\ell(f)
= -\sum_{i=1}^N
\left[
\log\!\left(t_i - f(x_i)\right)
+ \frac{\big(\ln(t_i - f(x_i)) - \mu\big)^2}{2\sigma^2}
\right]
+ C,
$$
where $C$ is a constant independent of $f$.

Hence, the negative log-likelihood (NLL) to minimize is
$$
\mathcal{L}_{\text{NLL}}(f)
= \sum_{i=1}^N
\left[
\log\!\left(t_i - f(x_i)\right)
+ \frac{\big(\ln(t_i - f(x_i)) - \mu\big)^2}{2\sigma^2}
\right],
$$
subject to the constraint
$$
t_i - f(x_i) > 0 \quad \forall i.
$$

---



## Q2. Choice of basis functions
Initially started with 
$
\{1, x, x^2, \dots, x^n\},
$ but even after setting degree upto 100, the predicted curve was nowhere close to true curve.


Later we used gaussian radial functions as basis.
Gaussian radial basis functions:
$$
\phi_j(x) = \exp\!\left(-\frac{(x-c_j)^2}{2s_j^2}\right)
$$
are well suited for mixture-of-Gaussians targets.


---

## Q3. 
The expected risk under squared loss is
$$
\mathbb{E}[L]
= \int \int (t - f(x))^2\, p(t,x)\, dt\, dx.
$$
Fix $x$. The conditional risk is
$$
\mathbb{E}[L \mid x]
= \int (t - f(x))^2\, p(t \mid x)\, dt.
$$

Differentiate with respect to $f(x)$:
$$
\frac{\partial}{\partial f(x)}
\int (t - f(x))^2\, p(t \mid x)\, dt
= -2 \int (t - f(x))\, p(t \mid x)\, dt.
$$

Setting the derivative to zero,
$$
\int (t - f(x))\, p(t \mid x)\, dt = 0.
$$

Thus,
$$
f(x) = \int t\, p(t \mid x)\, dt
= \mathbb{E}[t \mid x].
$$

$$
\boxed{f(x) = \mathbb{E}[t \mid x]}
$$
Under squared (L2) loss, the Bayes-optimal predictor is the conditional mean.


## Part 2 | Training

Create a class `SLR`, that takes as arguments --
* Learning rate
* Regularization coefficient
* Initial weights and biases
It has the following methods --
* `forward(self, x)`: Uses the basis functions followed by conditional mean of the noise distribution to compute estiamte of y. Note that for gaussian noise, this is equal to the mean of distribution, but not in this case. 
* `loss(self, t, y)`: Given the target value t, and estimated value y, it computes the loss between them. Don't forget to use the regularizer (assume $E_w(\mathbf{w}) = \mathbf{w}^T\mathbf{w}$). 
* `step(self)`: Performs one iteration of Stochastic Gradient Descent. 

Then, create another class `Trainer` that takes as argument --
* num_epochs - Number of iterations of SGD
It has the following method --
* `fit(self, model, train_data)`: `model` is an instance of `SLR` class. It calls relevant methods of model, to train the model for `num_epochs`. The train data can be generated by calling `generate_data` function.
* `predict(self, model, x)`: Called after fitting the model. This function simply evaluates the model to return the estimated y for input x.
  
Note that `train_data` is a matrix with `n_samples` rows and 2 columns (x, t) while x is a 1D array.


In [40]:
# TODO
import numpy as np

class SLR:

    def __init__(self, n_basis=3, learning_rate=1e-6, reg=0.0,
                 weights=None, bias=0.0, mu_init=0.0, sigma_init=0.1):
        self.n_basis = n_basis
        self.lr = float(learning_rate)
        self.reg = float(reg)
        if weights is None:
            self.w = np.zeros(self.n_basis, dtype=float)
        else:
            self.w = np.array(weights, dtype=float)
            if self.w.shape[0] != self.n_basis:
                raise ValueError("weights length must equal n_basis")
        self.b = float(bias)
        # Noise parameters (will be updated during training)
        self.mu = float(mu_init)
        self.sigma = float(sigma_init)  # standard deviation
        # numerical epsilon to avoid log(0)
        self._eps = 1e-9

    def _phi(self, x):
      
        x = np.asarray(x, dtype=float)
        if x.ndim == 0:
            x = x[np.newaxis]
       
        return np.vstack([x**(j+1) for j in range(self.n_basis)]).T

    def f(self, x):
        """Deterministic part f(x) = phi(x) @ w + b"""
        phi = self._phi(x)
        if phi.size == 0:
            return np.full_like(x, fill_value=self.b, dtype=float)
        return phi.dot(self.w) + self.b

    def expected_noise(self):
        """E[eta] for lognormal noise with current mu, sigma"""
        return np.exp(self.mu + 0.5 * (self.sigma**2))

    def forward(self, x):
        """
        Returns the conditional mean estimator y_hat(x) = E[t | x]
        for additive lognormal noise: f(x) + E[eta].
        x may be scalar or 1D array.
        """
        fx = self.f(x)
        c = self.expected_noise()
        return fx + c

    def loss(self, t, y):
        # using ridge regularisation
        t = np.asarray(t, dtype=float)
        y = np.asarray(y, dtype=float)
        mse = 0.5 * np.mean((t - y)**2)
        reg_term = 0.5 * self.reg * np.sum(self.w**2)
        return mse + reg_term

    def step(self, x_scalar, t_scalar):
        """
        One SGD update using a single sample (x_scalar, t_scalar).
        Minimizes squared error between t and conditional mean y = f(x) + E[eta]
        plus L2 regularization on w.

        Returns: scalar instantaneous loss (before regularization).
        """
        # compute phi and deterministic f(x)
        phi = self._phi(np.atleast_1d(x_scalar))[0]  # shape (n_basis,)
        fx = phi.dot(self.w) + self.b
        c = self.expected_noise()            # constant wrt parameters during this step
        y_hat = fx + c
        # y_hat = np.clip(y_hat, 0.0, 1e3)
        error = y_hat - t_scalar             # scalar: y - t  (consistent with loss 0.5*(t-y)^2)
        # gradients of 0.5*(t - y)^2 w.r.t w,b (note y depends on f but c is constant)
        # d/dw [0.5*(t - (f+c))^2] = -(t - y) * df/dw = (y - t) * phi
        grad_w = error * phi + self.reg * self.w  # include grad of 0.5*reg*||w||^2 => reg * w
        grad_b = error * 1.0                       # bias derivative
        # parameter update (SGD)
        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b
        # return instantaneous (sample) loss without regularizer for logging (optional)
        return 0.5 * (t_scalar - y_hat)**2

class Trainer:
    """
    Trainer for SLR using mini-batch SGD (here batch size = 1, i.e., pure SGD).
    After each epoch the trainer updates the model's mu and sigma based on
    residuals r_i = t_i - f(x_i) (not including expected noise).
    """
    def __init__(self, num_epochs=100, verbose=False):
        self.num_epochs = int(num_epochs)
        self.verbose = bool(verbose)

    def fit(self, model: SLR, train_data):
        """
        Train 'model' on train_data for num_epochs.
        train_data: array-like shape (n_samples, 2) where columns are (x, t)
        """
        data = np.asarray(train_data, dtype=float)
        X = data[:, 0]
        T = data[:, 1]
        N = X.shape[0]

        history = {'loss_epoch': []}
        for ep in range(self.num_epochs):
            # shuffle
            perm = np.random.permutation(N)
            epoch_loss = 0.0
            for i in perm:
                loss_i = model.step(X[i], T[i])
                epoch_loss += float(loss_i)
            epoch_loss /= float(N)
            # After finishing epoch, update noise parameters mu and sigma from residuals
            # Residuals r_i = t_i - f(x_i) (deterministic f, not including expected noise)
            f_vals = model.f(X)
            residuals = T - f_vals
            # ensure positivity for log; clip to a small epsilon (this enforces the model to be feasible)
            eps = model._eps
            residuals_clipped = np.maximum(residuals, eps)
            logs = np.log(residuals_clipped)
            model.mu = float(np.mean(logs))
            model.sigma = float(np.sqrt(np.mean((logs - model.mu)**2) + 0.0))  # MLE uses 1/N
            # record epoch loss using conditional mean predictions
            y_preds = model.forward(X)
            total_loss = model.loss(T, y_preds)
            history['loss_epoch'].append(total_loss)
            if self.verbose and (ep % max(1, self.num_epochs//10) == 0):
                print(f"Epoch {ep+1}/{self.num_epochs}: loss={total_loss:.6f}, mu={model.mu:.4f}, sigma={model.sigma:.4f}")
        return history

    def predict(self, model: SLR, x):
        """Return model.forward(x) for input 1D array x"""
        return model.forward(x)


In [ ]:

def get_mixture_density(x_vals):
    return 100 * sum(w * (1 / (s * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_vals - m) / s) ** 2) 
                for m, s, w in zip(means, stds, weights))

# instantiate model and trainer
model = SLR(n_basis=100, learning_rate=5e-5, reg=1e-3, mu_init=0.0, sigma_init=0.1)
trainer = Trainer(num_epochs=400, verbose=True)

# fit model
history = trainer.fit(model, data)

# prepare plotting grid and predictions
x_grid = np.linspace(0, 100, 400)
x_grid = x_grid / 100.0   # NORMALIZATION
y_pred_grid = trainer.predict(model, x_grid)
y_true_grid = get_mixture_density(x_grid*100.0)

# Plot: noisy training targets (scatter), predicted curve (line), and true noiseless curve (dashed)
plt.figure(figsize=(10,5))
plt.scatter(data[:,0], data[:,1], s=30, alpha=0.8, label='Train: noisy targets (t)')
#plt.scatter(data[:,0], model.f(data[:,0])+model.expected_noise(), s=30, alpha=0.8, label='Train: deterministic f(x)', color='orange', marker='x', zorder=4)
plt.plot(x_grid, y_pred_grid, linewidth=2, label='Model prediction (conditional mean)', zorder=3)
plt.plot(x_grid, y_true_grid, linestyle='--', linewidth=1.5, label='True mixture (noiseless)', zorder=2)
plt.xlabel('x')
plt.ylabel('y / t')
plt.title('Train targets and model predictions')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Print final estimated noise params
print(f"Estimated mu = {model.mu:.4f}, sigma = {model.sigma:.4f}")

USING GAUSSIAN RADIAL BASIS FUNCTIONS

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

means = [25, 58, 75]
stds  = [8, 12, 5]
weights = [0.25, 0.3, 0.2]

def get_mixture_density(x_vals):
    x = np.asarray(x_vals, dtype=float)
    return 100 * sum(w * (1 / (s * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - m) / s) ** 2) 
                   for m, s, w in zip(means, stds, weights))

def generate_data(n_samples, seed=None):
    """Generate (x_orig, t) pairs on original scale x in [0,100]."""
    if seed is not None:
        np.random.seed(seed)
    x_orig = np.linspace(0, 100, n_samples)
    y_true = get_mixture_density(x_orig)
    noise = np.random.lognormal(mean=0, sigma=0.1, size=n_samples)
    t = y_true + noise
    return x_orig, t


def scale_x(x_orig):
    """Scale original x in [0,100] to scaled domain [-1,1]."""
    return (np.asarray(x_orig, dtype=float) - 50.0) / 50.0

def unscale_x(x_scaled):
    """Inverse transform back to original scale."""
    return np.asarray(x_scaled, dtype=float) * 50.0 + 50.0

# ------------------------
# RBF features helper
# def rbf_features(x_scaled, centers_scaled, width):
#     """
#     Compute Gaussian RBF features for 1D x.
#     - x_scaled: array (N,)
#     - centers_scaled: array (K,)
#     - width: scalar (in same scaled units)
#     Returns: (N, K) matrix where column k is exp(-0.5*((x - c_k)/width)^2)
#     """
#     x = np.asarray(x_scaled)[:, None]           # (N,1)
#     c = np.asarray(centers_scaled)[None, :]     # (1,K)
#     Z = (x - c) / float(width)
#     return np.exp(-0.5 * (Z**2))
def rbf_features(x_scaled, centers_scaled, width):
    """
    Compute Gaussian RBF features for 1D x.
    Works for scalar or array inputs.
    """
    x = np.atleast_1d(x_scaled)[:, None]        # (N,1)
    c = np.asarray(centers_scaled)[None, :]    # (1,K)
    Z = (x - c) / float(width)
    return np.exp(-0.5 * (Z**2))

# ------------------------
# SLR class using RBF basis (SGD)
class SLR:

    def __init__(self, n_rbfs=15, learning_rate=1e-3, reg=1e-3,
                 centers_scaled=None, width=None, weights=None, bias=0.0,
                 mu_init=0.0, sigma_init=0.1):
        self.n_rbfs = int(n_rbfs)
        self.lr = float(learning_rate)
        self.reg = float(reg)
        # choose centers evenly in scaled domain [-1,1] if not provided
        if centers_scaled is None:
            self.centers = np.linspace(-1.0, 1.0, self.n_rbfs)
        else:
            self.centers = np.asarray(centers_scaled, dtype=float)
            assert self.centers.size == self.n_rbfs
        # width: default chosen to cover roughly spacing between centers
        if width is None:
            if self.n_rbfs > 1:
                spacing = float(self.centers[1] - self.centers[0])
                self.width = 1.5 * spacing   # slightly larger than spacing
            else:
                self.width = 0.5
        else:
            self.width = float(width)
        # parameters
        if weights is None:
            self.w = np.zeros(self.n_rbfs, dtype=float)
        else:
            self.w = np.asarray(weights, dtype=float)
        self.b = float(bias)
        # lognormal noise params (will be estimated from residuals)
        self.mu = float(mu_init)
        self.sigma = float(sigma_init)
        self._eps = 1e-9
        # clip predictions to avoid overflow
        self._pred_clip = 1e6

    def _phi(self, x_orig):
        """
        Build RBF feature matrix for original-scale x (we scale inside).
        Accepts scalar or array; returns (N, n_rbfs).
        """
        x_scaled = scale_x(x_orig)
        return rbf_features(x_scaled, self.centers, self.width)

    def f(self, x_orig):
        """Deterministic part f(x) = phi(x) @ w + b (x_orig may be scalar or array)"""
        Phi = self._phi(x_orig)  # (N,K)
        return Phi.dot(self.w) + self.b

    def expected_noise(self):
        """E[eta] for lognormal additive noise."""
        return np.exp(self.mu + 0.5 * (self.sigma**2))

    def forward(self, x_orig):
        """Return conditional mean estimate y_hat(x) = f(x) + E[eta]."""
        fx = self.f(x_orig)
        yhat = fx + self.expected_noise()
        # numeric safety
        return np.clip(yhat, -self._pred_clip, self._pred_clip)

    def loss(self, t, y):
        """Mean squared error (0.5*(t-y)^2 averaged) + regularization 0.5*reg*||w||^2"""
        t = np.asarray(t, dtype=float)
        y = np.asarray(y, dtype=float)
        mse = 0.5 * np.mean((t - y)**2)
        reg_term = 0.5 * self.reg * np.sum(self.w**2)
        return mse + reg_term

    def step(self, x_scalar, t_scalar):
        """
        One SGD update using single sample (x_scalar,t_scalar).
        Minimizes 0.5*(t - (f(x)+E[eta]))^2 + 0.5*reg*||w||^2 w.r.t. w and b.
        Note: we treat mu and sigma as fixed during the step; they are updated by Trainer.
        Returns instantaneous sample loss (0.5*(t-y)^2).
        """
        # compute features for this x (Phi is (1,K) -> take 0-th row)
        phi = self._phi(x_scalar)[0]      # shape (K,)
        fx = float(phi.dot(self.w) + self.b)
        c = self.expected_noise()         # scalar
        y_hat = fx + c
        # numeric safety: clip
        y_hat = float(np.clip(y_hat, -self._pred_clip, self._pred_clip))
        # error = y_hat - t  (we follow grad of 0.5*(t-y)^2)
        error = y_hat - float(t_scalar)
        # gradients: d/dw 0.5*(t - y)^2 = (y - t) * phi  ; plus reg gradient reg * w
        grad_w = error * phi + self.reg * self.w
        grad_b = error * 1.0
        # update parameters
        self.w -= self.lr * grad_w
        self.b -= self.lr * grad_b
        # return instantaneous loss (without regularizer)
        return 0.5 * (float(t_scalar) - y_hat)**2


class Trainer:
    def __init__(self, num_epochs=200, verbose=False):
        self.num_epochs = int(num_epochs)
        self.verbose = bool(verbose)

    def fit(self, model: SLR, train_data):
        """
        Train model using SGD (batch size = 1).
        - train_data: array-like (n_samples,2) where column0 is x_orig, column1 is t
        After each epoch we re-estimate mu and sigma from residuals r_i = t_i - f(x_i).
        """
        data = np.asarray(train_data, dtype=float)
        X_orig = data[:, 0]
        T = data[:, 1]
        N = X_orig.shape[0]
        history = {'loss_epoch': [], 'mu': [], 'sigma': []}
        for ep in range(self.num_epochs):
            perm = np.random.permutation(N)
            epoch_loss = 0.0
            for i in perm:
                loss_i = model.step(X_orig[i], T[i])
                epoch_loss += float(loss_i)
            epoch_loss /= float(N)
            # Recompute noise parameters from deterministic residuals: r_i = t_i - f(x_i)
            f_vals = model.f(X_orig)   # deterministic f(x) on original scale
            residuals = T - f_vals
            # force minimal positivity for log (since r_i should be > 0 for additive lognormal)
            residuals_clipped = np.maximum(residuals, model._eps)
            logs = np.log(residuals_clipped)
            model.mu = float(np.mean(logs))
            model.sigma = float(np.sqrt(np.mean((logs - model.mu)**2)))
            # record epoch metrics
            y_preds = model.forward(X_orig)
            total_loss = model.loss(T, y_preds)
            history['loss_epoch'].append(total_loss)
            history['mu'].append(model.mu)
            history['sigma'].append(model.sigma)
            if self.verbose and (ep % max(1, self.num_epochs//10) == 0):
                print(f"Epoch {ep+1}/{self.num_epochs}: loss={total_loss:.6f}, mu={model.mu:.4f}, sigma={model.sigma:.4f}")
        return history

    def predict(self, model: SLR, x_orig):
        """Predict on original-scale x values (array-like)"""
        return model.forward(x_orig)


np.random.seed(1)
x_orig, t = generate_data(n_samples=50, seed=None)  # original-scale data

# assemble train_data as requested: matrix (n_samples, 2)
train_data = np.column_stack((x_orig, t))

# create model (RBF basis)
model = SLR(n_rbfs=15, learning_rate=5e-3, reg=1e-3, width=0.2, mu_init=0.5, sigma_init=0.3)
trainer = Trainer(num_epochs=500, verbose=True)

# fit
history = trainer.fit(model, train_data)

# predictions on dense grid (original x)
x_grid = np.linspace(0, 100, 400)
y_pred_grid = trainer.predict(model, x_grid)
y_true_grid = get_mixture_density(x_grid)

# Plot 1: Predicted curve vs true mixture and training points
plt.figure(figsize=(10,5))
plt.scatter(x_orig, t, s=30, alpha=0.9, label='Train: noisy targets (t)')
plt.plot(x_grid, y_pred_grid, linewidth=2, label='Model prediction (conditional mean)')
plt.plot(x_grid, y_true_grid, linestyle='--', linewidth=1.5, label='True mixture (noiseless)')
plt.xlabel('x (original scale)')
plt.ylabel('y / t')
plt.title('Train targets and RBF-SLR predictions')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

# # Plot 2: Predicted vs Actual scatter (use predicted at training x)
# y_pred_train = trainer.predict(model, x_orig)
# plt.figure(figsize=(6,6))
# plt.scatter(t, y_pred_train, s=40, alpha=0.8)
# mn = min(t.min(), y_pred_train.min()); mx = max(t.max(), y_pred_train.max())
# plt.plot([mn,mx], [mn,mx], 'k--', linewidth=1)  # y=x reference
# plt.xlabel('Actual target t')
# plt.ylabel('Predicted y_hat')
# plt.title('Predicted vs Actual (train points)')
# plt.grid(alpha=0.3)
# plt.show()

# Print final model stats
print("Final mu, sigma:", model.mu, model.sigma)
print("First 8 RBF weights:", model.w[:8])


---

## Part 3 | Bias Variance Tradeoff

Now, that we have all components ready, we will train the regression model, and analyse its performance. 

### Q1
As yet, we have no way of optimizing `num_epochs`, `learning_rate` and initial weights and biases. I leave this as an open problem for you to experiment with. If you have trained models before, you might know that we using train-val loss graphs to find it. But this is covered in more detail in later chapters, so if you haven't done it before, just lookup on internet for an initial guess.  

### Q2
Try to recreate the graphs in **Figure 4.7**. You can use `x` in `Trainer.predict` as `np.linspace(0, 100, 1000)`, and plot that against x. Do this for 20 training datasets, each with `n_samples=50`. The average of the fits is simply the average of outputs of their `predict` functions. In this part, you can use the true function code to plot the green line in **Figure 4.7**. Try this for multiple values of regularization coefficient.

### Q3
To find the optimal regularization coefficient, plot the graph shown in **Figure 4.8**. The value of $\lambda$ at which $(\text{bias})^2 + \text{variance}$ is minimum, is the optimal choice. 

In [ ]:
# TODO
# ===============================
# Part 3: Bias-Variance Experiment (RBF features + closed-form ridge)
# ===============================
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# 1) Problem setup: mixture-of-Gaussians true function (original scale 0..100)
means = [25, 58, 75]
stds  = [8, 12, 5]
weights = [0.25, 0.3, 0.2]

def get_mixture_density(x_vals):
    x = np.asarray(x_vals, dtype=float)
    return 100 * sum(w * (1 / (s * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - m) / s) ** 2)
                   for m, s, w in zip(means, stds, weights))

# scaling helpers (original <-> scaled [-1,1])
def scale_x(x_orig):
    return (np.asarray(x_orig, dtype=float) - 50.0) / 50.0

def unscale_x(x_scaled):
    return np.asarray(x_scaled, dtype=float) * 50.0 + 50.0

# RBF features (works for scalar or array)
def rbf_features(x_orig, centers_orig, width_orig):
    """
    Build RBF features on ORIGINAL x scale.
    Internally scales x to [-1,1] (uniformly) and uses centers on original scale.
    """
    # scale inputs and centers to [-1,1]
    x = scale_x(x_orig)
    c = scale_x(centers_orig)
    x = np.atleast_1d(x)[:, None]    # (N,1)
    c = np.asarray(c)[None, :]      # (1,K)
    Z = (x - c) / float(width_orig) # width in scaled units
    return np.exp(-0.5 * (Z**2))    # (N,K)

# ridge closed-form solver (regularize weights only, not bias)
def ridge_closed_form(Phi, t, lam):
    """
    Phi: (N,K) design matrix (RBF features)
    t: (N,) targets
    lam: regularization scalar (lambda)
    Returns w (K,), b scalar.
    """
    N, K = Phi.shape
    A = np.hstack([Phi, np.ones((N,1))])  # (N, K+1)
    ATA = A.T.dot(A)
    reg = lam * np.eye(K+1)
    reg[-1, -1] = 0.0   # do not regularize bias
    theta = np.linalg.solve(ATA + reg, A.T.dot(t))
    w = theta[:-1]
    b = theta[-1]
    return w, b

# -------------------------------
# 2) Bias-Variance experiment parameters
n_datasets = 20         # number of independent training sets (L in Bishop)
n_samples = 50          # samples per training set
x_grid = np.linspace(0, 100, 1000)  # dense evaluation grid
y_true_grid = get_mixture_density(x_grid)

# RBF basis settings
K = 24                  # number of RBFs (like Bishop's experiment uses many bases)
centers_orig = np.linspace(0, 100, K)  # centers in original domain
# choose width in scaled domain (approx spacing)
centers_scaled = scale_x(centers_orig)
if K > 1:
    spacing = centers_scaled[1] - centers_scaled[0]
    width = 1.2 * spacing
else:
    width = 0.2

# noise model: zero-mean Gaussian (sigma_noise)
sigma_noise = 0.08  # tune: small noise relative to function amplitude
noise_variance = sigma_noise**2

# list of lambda values to sweep (ln lambda from -3 to +3)
ln_lams = np.linspace(-3.0, 3.0, 25)
lams = np.exp(ln_lams)

# For reproducibility: use a RandomState
rng = np.random.RandomState(0)

# -------------------------------
# 3) Generate many datasets and fit models (closed-form ridge) for each lambda
# We'll store predictions: preds[lam_index][dataset_index][:] on x_grid
preds = np.zeros((len(lams), n_datasets, x_grid.size))

for li, lam in enumerate(lams):
    for d in range(n_datasets):
        # generate dataset (uniform x positions + independent noise)
        # use same x positions (linspace) for every dataset to match Bishop
        x_train = np.linspace(0, 100, n_samples)
        y_train_true = get_mixture_density(x_train)
        noise = rng.normal(loc=0.0, scale=sigma_noise, size=n_samples)
        t_train = y_train_true + noise
        # build design and fit ridge
        Phi = rbf_features(x_train, centers_orig, width)
        w, b = ridge_closed_form(Phi, t_train, lam)
        # evaluate deterministic f on x_grid (no expected-noise addition here because noise mean is zero)
        Phi_grid = rbf_features(x_grid, centers_orig, width)
        f_grid = Phi_grid.dot(w) + b
        preds[li, d, :] = f_grid

# -------------------------------
# 4) Compute bias^2 and variance (average over x)
bias2 = np.zeros(len(lams))
variance = np.zeros(len(lams))
total = np.zeros(len(lams))   # bias^2 + variance + noise
for li in range(len(lams)):
    mean_pred = preds[li].mean(axis=0)       # average prediction over datasets (shape = len(x_grid))
    bias2[li] = np.mean((mean_pred - y_true_grid)**2)  # integrate over grid by averaging
    variance[li] = np.mean(preds[li].var(axis=0, ddof=1))
    total[li] = bias2[li] + variance[li] + noise_variance

# -------------------------------
# 5) Recreate Bishop-like figure for a few lambda values (overlay many fits, average)
# choose 3 ln lambda values to illustrate under/fit/overfit
ln_examples = [-3.0, 0.0, 3.0]
example_idx = [np.argmin(np.abs(ln_lams - le)) for le in ln_examples]

fig, axes = plt.subplots(len(example_idx), 2, figsize=(12, 3 * len(example_idx)))
for r, idx in enumerate(example_idx):
    li = example_idx[r]
    # left: overlay of many fits (thin lines)
    ax = axes[r, 0]
    for d in range(n_datasets):
        ax.plot(x_grid/100.0, preds[li, d, :], color='red', alpha=0.4, linewidth=0.9)
    ax.set_xlim(0, 1)
    ax.set_ylabel('t'); ax.set_xlabel('x (scaled 0..1)')
    ax.set_title(f'ln λ = {ln_lams[li]:.2f} (overlay fits)')
    # right: mean fit (red) vs true (green)
    ax2 = axes[r, 1]
    mean_pred = preds[li].mean(axis=0)
    ax2.plot(x_grid/100.0, y_true_grid, color='green', linestyle='--', label='true f(x)')
    ax2.plot(x_grid/100.0, mean_pred, color='red', label='mean fit', linewidth=1.5)
    ax2.set_xlim(0, 1)
    ax2.set_ylabel('t'); ax2.set_xlabel('x (scaled 0..1)')
    ax2.set_title('mean fit (red) vs true (green)')
    ax2.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(7,5))
plt.plot(ln_lams, bias2, color='red', label='bias^2')
plt.plot(ln_lams, variance, color='blue', label='variance')
plt.plot(ln_lams, bias2+variance, color='green', label='bias^2 + variance')
plt.plot(ln_lams, total, color='magenta', linestyle='--', label='total (incl. noise)')
plt.xlabel('ln λ')
plt.ylabel('Error (averaged over x)')
plt.title('Bias-Variance tradeoff vs ln λ')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


best_idx = np.argmin(bias2 + variance)
print("Optimal ln(lambda) (min bias^2+var) = {:.3f}, lambda = {:.4e}".format(ln_lams[best_idx], lams[best_idx]))
print("At optimum: bias^2 = {:.4e}, var = {:.4e}, noise_var = {:.4e}, total = {:.4e}".format(
    bias2[best_idx], variance[best_idx], noise_variance, total[best_idx]))

# generate many noisy test realizations (size 1000) and compute mean MSE across the fitted models.
x_test = np.linspace(0, 100, 1000)
y_true_test = get_mixture_density(x_test)
Phi_test = rbf_features(x_test, centers_orig, width)
empirical_test_mse = np.zeros(len(lams))
for li, lam in enumerate(lams):
    mses = []
    for d in range(n_datasets):
        # predicted (deterministic f)
        y_pred = preds[li, d, :]
        # generate one noisy test realization
        t_test = y_true_test + rng.normal(0.0, sigma_noise, size=y_true_test.shape)
        mses.append(np.mean((y_pred - t_test)**2))
    empirical_test_mse[li] = np.mean(mses)
plt.figure(figsize=(7,4))
plt.plot(ln_lams, empirical_test_mse, label='empirical test MSE')
plt.plot(ln_lams, bias2 + variance + noise_variance, '--', label='theoretical bias^2+var+noise')
plt.xlabel('ln λ'); plt.ylabel('MSE'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Empirical test MSE vs theoretical decomposition')
plt.show()
